In [38]:
!pip install -q torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

c:\Users\miras\OneDrive\Документы\GitHub\GPT-from-Scratch-PyTorch-\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: (''.join([itos[i] for i in l]))

print(encode('Hello, World!'))
print(decode([20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]))

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [7]:
print(len(text))
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

1115394


In [9]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [10]:
n = int(len(text)*0.9)
train_data = data[:n]
val_data = data[n:]

In [11]:
block_size = 8
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [12]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t:t+1]
  print(context, 'the next int', target)

tensor([18]) the next int tensor([47])
tensor([18, 47]) the next int tensor([56])
tensor([18, 47, 56]) the next int tensor([57])
tensor([18, 47, 56, 57]) the next int tensor([58])
tensor([18, 47, 56, 57, 58]) the next int tensor([1])
tensor([18, 47, 56, 57, 58,  1]) the next int tensor([15])
tensor([18, 47, 56, 57, 58,  1, 15]) the next int tensor([47])
tensor([18, 47, 56, 57, 58,  1, 15, 47]) the next int tensor([58])


In [13]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(0, len(data) - block_size -1, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

xb, yb = get_batch('train')
print(xb.shape)
xb, yb

torch.Size([4, 8])


(tensor([[53, 59,  6,  1, 58, 56, 47, 40],
         [49, 43, 43, 54,  1, 47, 58,  1],
         [13, 52, 45, 43, 50, 53,  8,  0],
         [ 1, 39,  1, 46, 53, 59, 57, 43]]),
 tensor([[59,  6,  1, 58, 56, 47, 40, 59],
         [43, 43, 54,  1, 47, 58,  1, 58],
         [52, 45, 43, 50, 53,  8,  0, 26],
         [39,  1, 46, 53, 59, 57, 43,  0]]))

In [14]:
n_embd = 32

In [ ]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape # (4, 8)
    tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
    x = tok_emb + pos_emb # (B, T, n_embd) + (T, n_embd) = (B, T, n_embd)
    logits = self.lm_head(x) 

    if targets is None:
      loss = None
    
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cropped = idx[:, -block_size:]
      logits, loss = self(idx_cropped)
      logits = logits[:, -1, :] # taking the last time step
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1) # (Batch_size, 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx
  
model = BigramLanguageModel()
m = model.to(device)
logits, loss = m(xb, yb)
loss

tensor(4.1905, grad_fn=<NllLossBackward0>)

In [44]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [45]:
for iter in range(max_iters):
  
  if iter % eval_interval == 0:
    losses = estimate_loss()
    print(f'Step {iter} losses: Train loss {losses['train']:.4f}, Val loss {losses['val']:.4f}')
  
  xb, yb = get_batch('train')
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

Step 0 losses: Train loss 4.4027, Val loss 4.4107
Step 300 losses: Train loss 3.1999, Val loss 3.2243
Step 600 losses: Train loss 2.9230, Val loss 2.9612
Step 900 losses: Train loss 2.7950, Val loss 2.8156
Step 1200 losses: Train loss 2.7544, Val loss 2.7341
Step 1500 losses: Train loss 2.6748, Val loss 2.6879
Step 1800 losses: Train loss 2.6790, Val loss 2.6384
Step 2100 losses: Train loss 2.6182, Val loss 2.6247
Step 2400 losses: Train loss 2.6042, Val loss 2.6223
Step 2700 losses: Train loss 2.5723, Val loss 2.6075


In [42]:
print(decode( m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=8)[0].tolist() ))


DS$.byNN


In [125]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones((T, T))) # creates triangle of 1s 
# wei = torch.zeros((T, T))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x) # (B, T, 16)
out = wei @ v # (B, T, 16)
out.shape

torch.Size([4, 8, 16])

In [124]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089